# t-score, log-Dice and Delta P

**Computational Text Analysis Pack · Module 05 — Concordance and Collocation**

*Technique:* The association measures the field actually reports, and their biases

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Sagaustus/adh-dh-notebooks/blob/main/03-comparing-corpora/05d_association_measures_completed.ipynb)

Data: `masakhanews.csv` — from the `data/` folder of this pack.

---

In [ ]:
# Setup — run this first.
import os, warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
from collections import Counter

# The data this notebook needs. It is downloaded from CourseHub automatically;
# uploading by hand is only a fallback if that server is unreachable.
NEEDED = ['masakhanews.csv']
DATA_URL = 'https://coursehub.africandigitalhumanities.ca/ide/pack-data/text/'

def _missing():
    return [f for f in NEEDED if not os.path.exists(f)]

missing = _missing()
if missing:
    # Fetch from CourseHub first, so nobody has to upload anything.
    import urllib.request, urllib.parse
    for _f in list(missing):
        try:
            urllib.request.urlretrieve(DATA_URL + urllib.parse.quote(_f), _f)
            print(f'downloaded {_f}')
        except Exception as _e:
            print(f'could not fetch {_f}: {_e}')
    missing = _missing()

if missing:
    try:
        from google.colab import files
    except ImportError:
        raise SystemExit('Place these next to the notebook: ' + ', '.join(missing))
    # Ask again until everything has arrived. The upload widget returns as soon
    # as you close it, so picking only some of the files would otherwise fail a
    # few lines below with a confusing FileNotFoundError.
    for _ in range(4):
        print('Select ALL of these at once (ctrl-click / cmd-click to multi-select):')
        print('   ' + ', '.join(missing))
        files.upload()
        missing = _missing()
        if not missing:
            break
        print('Still needed: ' + ', '.join(missing))
    if missing:
        raise SystemExit(
            'Missing: ' + ', '.join(missing) + '. Re-run this cell and select '
            'every file listed, or upload them with the folder icon on the left.')

df       = pd.read_csv('masakhanews.csv')
NIGERIAN = ['Hausa', 'Igbo', 'Yoruba', 'Nigerian Pidgin']
# Most of this pack runs on scikit-learn and the standard library. Also
# available: nltk, spacy, gensim and networkx (modules 14-19), Keras
# (case studies), and a GPU pod reached through /ide/api/neural/ for the
# neural modules — never called directly, so no token lives in a notebook.

print(f'Loaded. df has {len(df):,} rows and {df.shape[1]} columns.')

## What this notebook does

Notebook 05c computed PMI, Dice and log-likelihood. Three measures the field
reports constantly were missing, and each one is missing for a reason worth
knowing: they disagree with PMI in *systematic* directions.

- **t-score** favours high-frequency pairs — grammatical collocation.
- **log-Dice** is bounded and frequency-stable, which is why Sketch Engine makes
  it the default for comparing across corpora.
- **Delta P** is *directional*: it distinguishes "this word predicts that one"
  from "that word predicts this one". No other measure here does that.

In [ ]:
import re

def tokenise(text):
    """Words, lowercased, Unicode-aware.

    `\\w` matches Unicode letters, so ọ, à, ƙ, ɗ and ɓ survive. It does not
    match combining marks, and a tone mark on ọ or ẹ is always one, so the
    pattern adds the combining-accent range U+0300-U+036F; without it ọ̀rọ̀
    splits into ọ + rọ. The common shortcut `[a-z]+` throws away far more and
    quietly changes every count in this notebook — see notebook 03b.
    """
    return re.findall(r"[\w\u0300-\u036f]+", str(text).lower())

sample = "Àwọn ọmọ ìlú ti ṣe ìpàdé lórí ọ̀rọ̀ ẹ̀kọ́ — ƙungiyar ɗalibai ta yi taro"
print("sample     :", sample)
print("tokens     :", tokenise(sample))
print("bare \\w+   :", re.findall(r"\w+", sample.lower()))
print("naive [a-z]:", re.findall(r"[a-z]+", sample.lower()))
print()
print("The naive pattern splits words at every accented letter; bare \\w+")
print("splits them wherever a tone mark sits on ọ or ẹ.")

In [ ]:
NODE = "government"        # <- change this
WINDOW = 5

docs = [tokenise(t) for t in df["text"]]
unigram = Counter()
for d in docs:
    unigram.update(d)
N = sum(unigram.values())

pair = Counter()
for d in docs:
    for i, w in enumerate(d):
        if w != NODE:
            continue
        lo, hi = max(0, i - WINDOW), min(len(d), i + WINDOW + 1)
        for j in range(lo, hi):
            if j != i:
                pair[d[j]] += 1

print(f"node '{NODE}' occurs {unigram[NODE]:,} times in {N:,} tokens")
print(f"{len(pair):,} distinct collocates within +/-{WINDOW}")

## Five measures over the same counts

Every measure below reads the same four numbers: how often the pair occurs, how
often each word occurs alone, and the corpus size. They differ only in what they
do with them — which is precisely why they rank collocates differently.

In [ ]:
import math

def measures(w, joint, min_count=10):
    if unigram[w] < min_count or joint < min_count:
        return None
    f_node, f_w = unigram[NODE], unigram[w]
    expected = f_node * f_w * (2 * WINDOW) / N
    mi = math.log2(joint / expected)
    t = (joint - expected) / math.sqrt(joint)
    dice = 2 * joint / (f_node + f_w)
    log_dice = 14 + math.log2(dice)
    # Delta P in both directions: how much does seeing one shift the other?
    dp_node_to_w = joint / f_node - (f_w - joint) / (N - f_node)
    dp_w_to_node = joint / f_w - (f_node - joint) / (N - f_w)
    return {"collocate": w, "joint": joint, "freq": f_w, "MI": mi, "t_score": t,
            "log_Dice": log_dice, "dP_node_to_w": dp_node_to_w,
            "dP_w_to_node": dp_w_to_node}

rows = [m for w, j in pair.items() if (m := measures(w, j))]
coll = pd.DataFrame(rows)
print(f"{len(coll)} collocates pass the minimum-count filter")

In [ ]:
print("Top 10 by MI — rare, specific, often noisy:")
print(coll.nlargest(10, "MI")[["collocate", "joint", "freq", "MI"]]
      .to_string(index=False, float_format=lambda v: f"{v:.2f}"))
print()
print("Top 10 by t-score — frequent and grammatical:")
print(coll.nlargest(10, "t_score")[["collocate", "joint", "freq", "t_score"]]
      .to_string(index=False, float_format=lambda v: f"{v:.2f}"))
print()
print("Top 10 by log-Dice — the compromise Sketch Engine ships:")
print(coll.nlargest(10, "log_Dice")[["collocate", "joint", "freq", "log_Dice"]]
      .to_string(index=False, float_format=lambda v: f"{v:.2f}"))

## Delta P, and why direction matters

Every measure above is symmetric: the association of *A* with *B* equals the
association of *B* with *A*. But language is not symmetric. In English, seeing
*Kingdom* makes *United* very likely; seeing *United* barely predicts *Kingdom*.

Delta P separates the two, which is why it came into corpus linguistics from
associative-learning theory rather than from information theory.

In [ ]:
coll["asymmetry"] = (coll["dP_node_to_w"] - coll["dP_w_to_node"]).abs()
print(f"Most asymmetric collocates of '{NODE}':")
print(coll.nlargest(12, "asymmetry")[
        ["collocate", "joint", "dP_node_to_w", "dP_w_to_node"]]
      .to_string(index=False, float_format=lambda v: f"{v:.4f}"))
print()
print("A large dP_w_to_node with a small dP_node_to_w means: that word almost")
print(f"always appears near '{NODE}', but '{NODE}' appears in plenty of other")
print("company. The relationship runs one way.")

In [ ]:
# How much do the rankings actually disagree?
top = {m: set(coll.nlargest(25, m)["collocate"]) for m in
       ["MI", "t_score", "log_Dice"]}
print("Overlap in each pair's top 25:")
for a, b in [("MI", "t_score"), ("MI", "log_Dice"), ("t_score", "log_Dice")]:
    shared = len(top[a] & top[b])
    print(f"  {a:<9} vs {b:<9} {shared:>3} of 25 shared")
print()
print("These are not interchangeable. Reporting 'collocates' without naming the")
print("measure and the window makes a result impossible to reproduce.")

### 🔧 Your turn 1

Set NODE to a word from a different topic — 'health' or 'church' — and compare which measure gives the most readable collocate list.

<details markdown="1">
<summary><b>What you should have seen</b> — click to expand</summary>

For most nodes log-Dice gives the list a reader finds most useful: MI fills up
with rare co-occurrences that are often typos or names, and t-score fills up
with *the*, *of* and *to*. That is the practical reason log-Dice is the Sketch
Engine default. It is not that the others are wrong — MI is answering "which
words are unusually specific to this node" and t-score "which co-occurrences am
I most confident are not chance". They are different questions.

</details>